In [101]:
import numpy as np
from collections import Counter

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" # GPT-2 regex 

In [102]:
text = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"

# Let's initialize the vocabulary 

vocabulary = {}
for i in range(256):
    key = bytes([i])
    value = i 
    vocabulary[key] = value 

# Let's do the pretokenization 

splitted_text = text.split(" ")
frequency_table = {}

for word in splitted_text:
    key = []
    encoding = word.encode('utf-8')
    for byte in encoding:
        key.append(bytes([byte]))
    key = tuple(key)
    value = frequency_table.get(key, 0)
    frequency_table[key] = value + 1

print(f"Frequency table at step 0 is ", frequency_table)
# Let's do the iterative merging 

# Implement the naive approach firts. Given the 
# frequency_table generate the pair_table and find max_pair

def find_max_pair(freq_table: dict) -> (tuple, dict):
    # We count all the pairs
    pair_table = {}

    for key in freq_table:
        length = len(key)
        for i in range(length - 1):
            pair_key = tuple((key[i], key[i + 1]))
            pair_value = pair_table.get(pair_key, 0)
            pair_table[pair_key] = pair_value + freq_table[key] # current value + occurence of the word where the pair is
    
    # Now we find the maximum pair 

    max_pair = None
    max_occurence = -1
    for pair in pair_table:
        if pair_table[pair] >= max_occurence:
            max_occurence = pair_table[pair]
            max_pair = pair

    return max_pair, pair_table

def update_freq_table(freq_table: dict, max_pair: tuple) -> dict:
    new_freq_table = {}

    for key in freq_table:
        value = freq_table[key]
        if max_pair[0] not in key or max_pair[1] not in key:
            new_freq_table[key] = value
        else:
            new_key = []
            i = 0
            while i < len(key) - 1:
                if key[i] == max_pair[0] and key[i + 1] == max_pair[1]:
                    new_key.append(max_pair[0] + max_pair[1])
                    i += 2
                else:
                    new_key.append(key[i])
                    i += 1
            if i == len(key) - 1:
                new_key.append(key[i])
            new_key_tuple = tuple(new_key)
            new_freq_table[new_key_tuple] = value

    return new_freq_table

N = 8

for i in range(N):
    # Find the max_pair
    curr_max_pair, pair_table = find_max_pair(frequency_table)

    # Add it to the dictionary
    vocabulary[curr_max_pair[0] + curr_max_pair[1]] = len(vocabulary)

    # Update the frequency_table
    frequency_table = update_freq_table(frequency_table, curr_max_pair)
    # print(f"The merged pair is {curr_max_pair}")
    print(f"Frequency table at step {i + 1} is ", frequency_table)

# print(vocabulary)

Frequency table at step 0 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'e', b's', b't'): 6, (b'w', b'i', b'd', b'e', b's', b't'): 3}
Frequency table at step 1 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'e', b'st'): 6, (b'w', b'i', b'd', b'e', b'st'): 3}
Frequency table at step 2 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 3 is  {(b'l', b'ow'): 5, (b'l', b'ow', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 4 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 5 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'e', b'west'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 6 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'ewest'): 6, (b'w', b'i', b'd', b'est')

In [103]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [104]:
re.findall(PAT,"some text I'll pretokenize")

['some', ' text', ' I', "'ll", ' pretokenize']

In [107]:
# Now, let's download or tokenizer and do some experiments.
from tokenizer import Tokenizer
from data_preprocessing import get_batch
vocab_path = '../output/vocab_tiny_stories.json'
merges_path = '../output/merges_list.txt'
special_tokens = ['<|endoftext|>']
custom_tokenizer = Tokenizer.from_files(vocab_filepath=vocab_path, merges_filepath=merges_path, special_tokens=special_tokens)

import tiktoken
from tokenizer import Tokenizer
gpt2_tokenizer = tiktoken.get_encoding('gpt2')
gpt2_tokenizer.encode("HELLO")

In [108]:
# We sample 10 documents from TinyStories Dataset and find the compression ratio.
# I think it should be around 5. 

with open('../output/TinyStories.txt', 'r') as file:
    text = file.read(100000)

In [109]:
ids = custom_tokenizer.encode(text=text)
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

ids = gpt2_tokenizer.encode(text, allowed_special = {'<|endoftext|>'})
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

100%|██████████| 251/251 [00:00<00:00, 852.24it/s]


The compression ratio is 4.181 bytes per id
The compression ratio is 4.107 bytes per id


In [110]:
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

The compression ratio is 4.107 bytes per id


In [69]:
# Training Accounting
B = 64
H = 12
T = 256
d = 96
L = 12
vocab = 10000
Attn = B * H * T * T * 4
QKV = B * T * d * 4 * 3
RMS = B * T * d * 4
FFN = B * T * (4 * d) * 4
RS = B * T * d * 4
Embed = B * T * d * 4
Output = B * T * vocab * 4
TOT = ((Attn + QKV + RMS + FFN + RS) * L + Embed + Output)*2
TOT/1e9

7.514095616